In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import time

# 1. Environment Configuration
# Load variables with override enabled to ensure the most recent data is used
load_dotenv(override=True)

# 2. Database Connection Setup
# Constructing the connection string and initializing the SQLAlchemy engine
DATABASE_URL = f"postgresql://{os.getenv('user')}:{os.getenv('password')}@{os.getenv('host')}:{os.getenv('port')}/{os.getenv('dbname')}"
engine = create_engine(DATABASE_URL)

try:
    print("Loading CSV into memory...")
    df = pd.read_csv('../1_data_extraction/data/historical_assets.csv')

    # Data Cleaning and Column Mapping
    # Standardizing column names to match the database schema
    df = df.rename(columns={
        'Open': 'open_price', 'High': 'high_price', 
        'Low': 'low_price', 'Close': 'close_price', 'Volume': 'volume'
    })
    
    # Standardize date format to YYYY-MM-DD
    df['date'] = pd.to_datetime(df['date']).dt.date
    
    # Filter for target columns only
    valid_columns = ['ticker', 'date', 'open_price', 'high_price', 'low_price', 'close_price', 'volume']
    df_final = df[valid_columns]

    # Batch Processing Configuration
    # A chunk size of 500 is optimal for stability when using transaction poolers
    total_rows = len(df_final)
    chunk_size = 500 

    print(f"Starting upload of {total_rows} rows in chunks of {chunk_size}...")
    start_time = time.time()

    # 4. Data Loading (Chunk-based ingestion)
    # Combining 'method=multi' with 'chunksize' ensures optimal stability and performance
    df_final.to_sql(
        'price_history', 
        engine, 
        if_exists='append', 
        index=False, 
        method='multi', 
        chunksize=chunk_size
    )

    end_time = time.time()
    duration = (end_time - start_time) / 60
    print(f"\nUpload completed successfully in {duration:.2f} minutes!")

except Exception as e:
    print(f"\nCritical Error: {e}")

Loading CSV into memory...
Starting upload of 338936 rows in chunks of 500...

Upload completed successfully in 2.20 minutes!
